In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from itertools import product


In [2]:
# Carregar o arquivo Excel com os dados
df = pd.read_csv("Base Completa.csv", sep=";")
df.dtypes


Evento            int64
Especialista      int64
Tipo             object
Nota              int64
r_menos_um      float64
r                 int64
r_mais_um       float64
f_r-1            object
f_r              object
f_1              object
dtype: object

In [8]:
# Lista de colunas numéricas a tratar
colunas_numericas = ['r_menos_um', 'r', 'r_mais_um', 'f_r-1', 'f_r', 'f_1']

# Tratamento das colunas: converter vírgula para ponto, depois para float, e preencher NaN com 0.0
for col in colunas_numericas:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
    
df.head(n=10)


,Evento,Especialista,Tipo,Nota,r_menos_um,r,r_mais_um,f_r-1,f_r,f_1
0,1,1,O,8,7.0,8,9.0,0.106507,0.786986,0.106507
1,1,2,O,7,6.0,7,8.0,0.106507,0.786986,0.106507
2,1,3,O,8,7.0,8,9.0,0.106507,0.786986,0.106507
3,1,1,D,7,6.0,7,8.0,0.106507,0.786986,0.106507
4,1,2,D,7,6.0,7,8.0,0.106507,0.786986,0.106507
5,1,1,S,8,7.0,8,9.0,0.106507,0.786986,0.106507
6,1,2,S,8,7.0,8,9.0,0.106507,0.786986,0.106507
7,1,3,S,8,7.0,8,9.0,0.106507,0.786986,0.106507
8,2,1,O,10,9.0,10,0.0,0.119203,0.880797,0.000000
9,2,2,O,10,9.0,10,0.0,0.119203,0.880797,0.000000


In [6]:
def calcular_rpn(df_sub):
    massa_total = defaultdict(float)
    especialistas = df_sub['Especialista'].nunique()

    for _, row in df_sub.iterrows():
        massa_total[row['r_menos_um']] += row['f_r-1']
        massa_total[row['r']] += row['f_r']
        massa_total[row['r_mais_um']] += row['f_1']
    
    print(massa_total)

    # Dividir pela quantidade de especialistas para obter média
    for k in massa_total:
        massa_total[k] /= especialistas

    # Calcular valor esperado
    rpn_esperado = sum(k * v for k, v in massa_total.items())
    return round(rpn_esperado, 4)


In [7]:

resultados=[]
for (evento, tipo), grupo in df.groupby(['Evento', 'Tipo']):
    rpn = calcular_rpn(grupo)
    resultados.append({'Evento': evento, 'Tipo': tipo, 'RPN_esperado': rpn})

# Converter em DataFrame final
df_resultado = pd.DataFrame(resultados)
#df_resultado.to_csv("resultado_rpn.csv")

df_resultado

defaultdict(<class 'float'>, {6.0: 0.213013958, 7: 1.573972084, 8.0: 0.213013958})
defaultdict(<class 'float'>, {7.0: 1.0, 8: 1.680479063, 9.0: 0.213013958, 6.0: 0.106506979})
defaultdict(<class 'float'>, {7.0: 0.319520937, 8: 2.360958126, 9.0: 0.319520937})
defaultdict(<class 'float'>, {2.0: 0.106506979, 3: 0.893493021, 4.0: 1.0, 5.0: 0.893493021, 6.0: 0.106506979})
defaultdict(<class 'float'>, {9.0: 0.34491282300000004, 10: 1.761594156, 0.0: 0.0, 7.0: 0.106506979, 8: 0.786986042})
defaultdict(<class 'float'>, {6.0: 0.893493021, 7: 0.893493021, 8.0: 0.213013958, 9: 0.786986042, 10.0: 0.106506979, 5.0: 0.106506979})
defaultdict(<class 'float'>, {4.0: 0.106506979, 5: 1.0, 6.0: 1.680479063, 7.0: 0.213013958})
defaultdict(<class 'float'>, {9.0: 0.33221688, 10: 0.880797078, 0.0: 0.0, 7.0: 0.213013958, 8: 1.573972084})
defaultdict(<class 'float'>, {4.0: 0.893493021, 5: 0.893493021, 6.0: 0.106506979, 3.0: 0.106506979})
defaultdict(<class 'float'>, {4.0: 0.213013958, 5: 1.573972084, 6.0: 0.21

,Evento,Tipo,RPN_esperado
0,1,D,7.0000
1,1,O,7.6667
2,1,S,8.0000
3,2,D,4.0000
4,2,O,9.2539
...,...,...,...
102,35,O,5.0000
103,35,S,8.3333
104,36,D,8.6269
105,36,O,8.2936


In [9]:
df_pivot = df_resultado.pivot(index='Evento', columns='Tipo', values= 'RPN_esperado')
df_pivot.columns = [F"RPN_{col}" for col in df_pivot.columns]
df_pivot.reset_index(inplace=True)

In [10]:
df_pivot['resultado'] = df_pivot['RPN_D'] * df_pivot['RPN_O'] * df_pivot['RPN_S'] 

df_pivot['resultado_filled'] = df_pivot['resultado'].fillna(-1)  # ou use 0 ou outro valor adequado

df_pivot['Ranking'] = df_pivot['resultado_filled'].rank(ascending=False, method='min').astype(int)

df_pivot = df_pivot.drop(columns='resultado_filled')

colunas_ordenadas = ['Ranking', 'Evento', 'RPN_O', 'RPN_D', 'RPN_S', 'resultado']

df_pivot = df_pivot[colunas_ordenadas]

df_pivot

,Ranking,Evento,RPN_O,RPN_D,RPN_S,resultado
0,4,1,7.6667,7.0000,8.0000,429.335200
1,14,2,9.2539,4.0000,7.3333,271.446499
2,19,3,8.6269,5.6667,4.5000,219.987244
3,7,4,8.2936,5.0000,8.9603,371.565720
4,9,5,9.4404,4.0000,8.9404,337.603809
5,10,6,8.2936,4.6667,7.9603,308.093406
6,8,7,9.4404,5.0000,7.5000,354.015000
7,26,8,6.6667,3.5000,6.0000,140.000700
8,31,9,9.2936,1.1192,5.6667,58.941597
9,18,10,7.0000,5.5000,6.3333,243.832050


In [17]:
# Reorganiza a ordem das colunas


# Ordena os dados pela coluna desejada (exemplo: RPN_D)
df_pivot = df_pivot.sort_values(by='resultado', ascending=False)  # ou ascending=True para ordem crescente



# Exibe resultado
print(df_pivot)


    Evento   RPN_O   RPN_D   RPN_S   resultado
35      36  8.2936  8.6269  9.2936  664.939030
32      33  9.8808  6.3333  9.2539  579.091208
29      30  7.9603  7.0000  8.6667  482.926724
0        1  7.6667  7.0000  8.0000  429.335200
33      34  8.2936  6.3333  7.3333  385.187866
20      21  5.0000  9.4404  8.0000  377.616000
3        4  8.2936  5.0000  8.9603  371.565720
6        7  9.4404  5.0000  7.5000  354.015000
4        5  9.4404  4.0000  8.9404  337.603809
5        6  8.2936  4.6667  7.9603  308.093406
34      35  5.0000  7.0000  8.3333  291.665500
31      32  6.5000  5.6667  7.5000  276.251625
12      13  8.6667  4.3333  7.3333  275.405096
1        2  9.2539  4.0000  7.3333  271.446499
15      16  3.6667  9.4404  7.6667  265.383700
30      31  5.5000  6.0000  7.9603  262.689900
17      18  5.3333  5.6667  8.3333  251.850752
9       10  7.0000  5.5000  6.3333  243.832050
2        3  8.6269  5.6667  4.5000  219.987244
10      11  4.5000  6.0000  6.3333  170.999100
26      27  7

In [22]:
# Carregar o arquivo Excel com os dados
df = pd.read_csv("Base Completa.csv", sep=";")

df.dtypes

Evento            int64
Especialista      int64
Tipo             object
Nota              int64
r_menos_um      float64
r                 int64
r_mais_um       float64
f_r-1            object
f_r              object
f_1              object
dtype: object

In [23]:
# Lista de colunas numéricas a tratar
colunas_numericas = ['r_menos_um', 'r', 'r_mais_um', 'f_r-1', 'f_r', 'f_1']

# Tratamento das colunas: converter vírgula para ponto, depois para float, e preencher NaN com 0.0
for col in colunas_numericas:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
    
df.sample(n=16)

,Evento,Especialista,Tipo,Nota,r_menos_um,r,r_mais_um,f_r-1,f_r,f_1
196,26,1,D,3,2.0,3,4.0,0.106507,0.786986,0.106507
255,34,1,O,10,9.0,10,0.0,0.119203,0.880797,0.000000
1,1,2,O,7,6.0,7,8.0,0.106507,0.786986,0.106507
236,31,3,D,6,5.0,6,7.0,0.106507,0.786986,0.106507
175,23,1,D,1,0.0,1,2.0,0.000000,0.880797,0.119203
76,10,2,S,5,4.0,5,6.0,0.106507,0.786986,0.106507
204,27,1,D,1,0.0,1,2.0,0.000000,0.880797,0.119203
258,34,1,D,5,4.0,5,6.0,0.106507,0.786986,0.106507
221,29,2,D,2,1.0,2,3.0,0.106507,0.786986,0.106507
30,4,1,S,10,9.0,10,0.0,0.119203,0.880797,0.000000


In [11]:
def calcular_coeficiente_K(m1, m2):
    conflito = 0
    interseccao = 0

    for (k1, v1), (k2, v2) in product(m1.items(), m2.items()):
        if k1 == k2:
            interseccao += v1 * v2
        else:
            conflito += v1 * v2

    K = interseccao / (interseccao + conflito) if (interseccao + conflito) != 0 else 0
    return K

def extrair_funcao_massa(row):
    return {
        int(row['r_menos_um']): row['f_r-1'],
        int(row['r']): row['f_r'],
        int(row['r_mais_um']): row['f_1']
    }


In [12]:
resultados = []

for (evento, tipo), grupo in df.groupby(['Evento', 'Tipo']):
    funcoes = grupo.apply(extrair_funcao_massa, axis=1).tolist()
    if len(funcoes) < 2:
        continue
    total_k = 0
    pares = 0
    for i in range(len(funcoes)):
        for j in range(i+1, len(funcoes)):
            k = calcular_coeficiente_K(funcoes[i], funcoes[j])
            total_k += k
            pares += 1
    k_medio = total_k / pares if pares > 0 else np.nan
    resultados.append({'Evento': evento, 'Tipo': tipo, 'K': round(k_medio, 4)})

# Resultados finais
df_K = pd.DataFrame(resultados)
df_K

,Evento,Tipo,K
0,1,D,0.6420
1,1,O,0.3258
2,1,S,0.6420
3,2,D,0.1155
4,2,O,0.2718
...,...,...,...
102,35,O,0.6420
103,35,S,0.3258
104,36,D,0.0663
105,36,O,0.0601


In [13]:
df_K_pivot = df_K.pivot(index='Evento', columns='Tipo', values= 'K')

df_K_pivot.columns = [F"K_{col}" for col in df_K_pivot.columns]

df_K_pivot.reset_index(inplace=True)

df_K_pivot.to_csv('coeficientes.csv')

df_K_pivot





,Evento,K_D,K_O,K_S
0,1,0.6420,0.3258,0.6420
1,2,0.1155,0.2718,0.0597
2,3,0.3258,0.2225,0.1676
3,4,0.6420,0.0601,0.1227
4,5,0.0113,0.1876,0.0127
5,6,0.3258,0.0601,0.0080
6,7,0.1155,0.1876,0.1676
7,8,0.1676,0.0597,0.6420
8,9,0.7900,0.3391,0.0597
9,10,0.1676,0.1155,0.2216


In [43]:
# Supondo que os DataFrames df_pivot e df_K_pivot já estejam carregados

# Merge (junção) pelo campo 'Evento'
df_merged = pd.merge(df_pivot, df_K_pivot, on='Evento', how='inner')

# Reorganiza e renomeia colunas
df_merged = df_merged.rename(columns={'resultado': 'RPNf'})

# Reordena as colunas conforme solicitado
df_final = df_merged[['Evento', 'K_O', 'RPN_O', 'K_D', 'RPN_D', 'K_S', 'RPN_S', 'RPNf', 'Ranking']]

# Visualiza a tabela final
df_final
df_final.to_excel("tabela_completa_final.xlsx", index=False)
